# Retrieval Snapshot Viewer

Visualizes oracle retrieval results across training epochs for fixed queries,
with a **CLIP baseline** (no condition) column for direct comparison.

**Data source**: `<experiment_dir>/retrieval_snapshots/`

- `metadata.pt` — captions, image paths, GT maps, CLIP baseline top-K (saved once)
- `epoch_XXXX.pt` — oracle top-K for that epoch (saved every evaluation interval)


In [ ]:
# ── CONFIG ──────────────────────────────────────────────────────────────────
EXPERIMENT_DIR = "/project/CoSiR/res/CoSiR_Experiment/impressions/20260430_134913_CoSiR_Experiment"

N_RETRIEVE = 5  # how many top results to show per column
N_EPOCHS_SHOW = 5  # max epoch columns (None = all); CLIP baseline is always shown
TOP_N_QUERIES = 5  # how many most-improved queries to highlight

# "top1" : improvement based on rank of the first (best) GT hit
# "mean" : improvement based on mean rank across ALL GT hits in the retrieved list
IMPROVEMENT_METRIC = "mean"

# None = auto-detect from snapshot; set "txt" or "img" to override
# Use this when the snapshot predates the combine_side key being saved.
COMBINE_SIDE: str | None = "img"
# ────────────────────────────────────────────────────────────────────────────

In [ ]:
import os, glob, textwrap
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image

snap_dir = os.path.join(EXPERIMENT_DIR, "retrieval_snapshots")

# ── Load metadata ──────────────────────────────────────────────────────────
meta = torch.load(os.path.join(snap_dir, "metadata.pt"), map_location="cpu")
captions = meta["captions"]
image_paths = meta["image_paths"]
ittmap = meta["image_to_text_map"]  # [N_img, cpi]
ttimap = meta["text_to_image_map"]  # [N_txt]
cpi = meta["captions_per_image"]
clip_base = meta["clip_baseline"]  # {"i2t": ..., "t2i": ...}

# ── Load epoch snapshots ───────────────────────────────────────────────────
snap_files = sorted(glob.glob(os.path.join(snap_dir, "epoch_*.pt")))
snaps = [torch.load(f, map_location="cpu") for f in snap_files]
all_epochs = [s["epoch"] for s in snaps]

if N_EPOCHS_SHOW is not None and len(snaps) > N_EPOCHS_SHOW:
    idx_sel = np.round(np.linspace(0, len(snaps) - 1, N_EPOCHS_SHOW)).astype(int)
    snaps = [snaps[i] for i in idx_sel]
    all_epochs = [s["epoch"] for s in snaps]

# Detect combine_side: COMBINE_SIDE config > epoch snap > metadata > default "txt"
combine_side = (
    COMBINE_SIDE
    or snaps[-1].get("combine_side")
    or meta.get("combine_side", "txt")
)

print(f"Loaded {len(snaps)} epoch snapshots: {all_epochs}")
print(
    f"Dataset: {meta['n_images']} images, {meta['n_texts']} texts, {cpi} captions/image"
)
print(f"CLIP baseline present: {'clip_baseline' in meta}")
print(f"combine_side: {combine_side}  (override with COMBINE_SIDE in cfg cell)")

In [ ]:
image_paths[0]

In [ ]:
# Change the image path prefix from the cluster path to the local path
prefix = "/var/scratch/wding/Dataset//redcaps_test/"
new_prefix = "/data/PDD/"

image_paths = [item.replace(prefix, new_prefix) for item in image_paths]

In [ ]:
image_paths[2]

## Helper: shared visualisation functions

`show_i2t` and `show_t2i` accept an optional `snaps_override` list so the
most-improved cells can pass only `[snaps[-1]]` (CLIP + last epoch only).


In [ ]:
def show_i2t(query_image_idx: int, n_retrieve: int = N_RETRIEVE, snaps_override=None):
    """Show top-K retrieved captions: CLIP baseline + one column per epoch.

    txt-side: conditioned texts are the gallery; image queries them.
    img-side: conditioned image is the query; it queries the text gallery.
    Both produce the same top-K text output format.
    """
    epoch_snaps = snaps_override if snaps_override is not None else snaps

    columns = [
        {
            "label": "CLIP\n(no cond)",
            "top_k": clip_base["i2t"]["top_k"][query_image_idx].tolist(),
            "is_gt": clip_base["i2t"]["is_gt"][query_image_idx].tolist(),
        }
    ]
    for snap in epoch_snaps:
        columns.append(
            {
                "label": f"Epoch {snap['epoch']}",
                "top_k": snap["i2t"]["top_k"][query_image_idx].tolist(),
                "is_gt": snap["i2t"]["is_gt"][query_image_idx].tolist(),
            }
        )

    n_rows = n_retrieve
    n_cols = len(columns) + 1  # +1 for query image
    img_w, col_w, row_h = 2.4, 3.2, 1.1

    fig = plt.figure(
        figsize=(img_w + col_w * len(columns), row_h * n_rows + 0.6),
        constrained_layout=True,
    )
    gs = gridspec.GridSpec(
        n_rows, n_cols, figure=fig, width_ratios=[img_w] + [col_w] * len(columns)
    )

    ax_img = fig.add_subplot(gs[:, 0])
    try:
        ax_img.imshow(Image.open(image_paths[query_image_idx]).convert("RGB"))
    except Exception:
        ax_img.text(
            0.5, 0.5, f"img {query_image_idx}", ha="center", va="center",
            transform=ax_img.transAxes, fontsize=7,
        )
    ax_img.set_title(f"Query\nimage {query_image_idx}", fontsize=8, pad=2)
    ax_img.axis("off")

    for col, col_data in enumerate(columns):
        for row in range(n_rows):
            ax = fig.add_subplot(gs[row, col + 1])
            if row < len(col_data["top_k"]):
                tidx = col_data["top_k"][row]
                gt = col_data["is_gt"][row]
                cap = captions[tidx] if tidx < len(captions) else "?"
                words = cap.split()
                if len(words) > 25:
                    cap = " ".join(words[:25]) + "…"
                wrapped = "\n".join(textwrap.wrap(cap, width=30))
                bg = "#b6f5b0" if gt else "white"
                ax.set_facecolor(bg)
                ax.text(
                    0.5, 0.5, f"#{row+1}\n{wrapped}", ha="center", va="center",
                    fontsize=6.5, transform=ax.transAxes,
                    bbox=dict(boxstyle="square,pad=0", facecolor=bg, linewidth=0),
                )
            ax.axis("off")
            if row == 0:
                ax.set_title(col_data["label"], fontsize=8, pad=2)

    fig.suptitle(
        f"I2T — image query {query_image_idx}  (green = GT)  [{combine_side}-side]",
        fontsize=9,
    )
    plt.show()


def show_t2i(query_text_idx: int, n_retrieve: int = N_RETRIEVE, snaps_override=None):
    """Show top-K retrieved images: CLIP baseline + one column per epoch.

    txt-side: conditioned text is the query; it queries the image gallery.
    img-side: text queries the conditioned image gallery.
    Both produce the same top-K image output format.
    """
    epoch_snaps = snaps_override if snaps_override is not None else snaps

    columns = [
        {
            "label": "CLIP\n(no cond)",
            "top_k": clip_base["t2i"]["top_k"][query_text_idx].tolist(),
            "is_gt": clip_base["t2i"]["is_gt"][query_text_idx].tolist(),
        }
    ]
    for snap in epoch_snaps:
        columns.append(
            {
                "label": f"Epoch {snap['epoch']}",
                "top_k": snap["t2i"]["top_k"][query_text_idx].tolist(),
                "is_gt": snap["t2i"]["is_gt"][query_text_idx].tolist(),
            }
        )

    n_rows = n_retrieve
    n_cols = len(columns)
    img_w, img_h = 1.8, 1.5

    query_cap = (
        captions[query_text_idx] if query_text_idx < len(captions) else str(query_text_idx)
    )
    gt_img_idx = ttimap[query_text_idx].item()
    q_short = "\n".join(textwrap.wrap(query_cap[:120], width=80))
    n_caption_lines = len(q_short.split("\n")) + 1
    top_margin = 0.4 + 0.22 * n_caption_lines

    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(img_w * n_cols, img_h * n_rows + top_margin),
        squeeze=False,
    )
    fig.subplots_adjust(top=1 - top_margin / (img_h * n_rows + top_margin))

    for col, col_data in enumerate(columns):
        for row in range(n_rows):
            ax = axes[row, col]
            if row < len(col_data["top_k"]):
                iidx = col_data["top_k"][row]
                gt = col_data["is_gt"][row]
                try:
                    ax.imshow(Image.open(image_paths[iidx]).convert("RGB"))
                except Exception:
                    ax.text(0.5, 0.5, f"img {iidx}", ha="center", va="center",
                            transform=ax.transAxes, fontsize=7)
                ax.set_xlabel(f"#{row+1}  img{iidx}", fontsize=6)
                if gt:
                    for spine in ax.spines.values():
                        spine.set_edgecolor("limegreen"); spine.set_linewidth(3)
            ax.set_xticks([]); ax.set_yticks([])
            if row == 0:
                ax.set_title(col_data["label"], fontsize=8, pad=2)

    fig.suptitle(
        f"T2I — text query {query_text_idx}  (green border = GT img {gt_img_idx})  "
        f"[{combine_side}-side]\n\"{q_short}\"",
        fontsize=8,
    )
    plt.show()

## Image → Text (I2T) — all epochs


In [ ]:
for qidx in range(5):
    show_i2t(query_image_idx=qidx)

## Text → Image (T2I) — all epochs


In [ ]:
for qidx in range(5):
    show_t2i(query_text_idx=qidx)

## R@K across epochs (vs CLIP baseline)


In [ ]:
def compute_recall(is_gt_tensor, k_vals=(1, 5, 10)):
    row = {}
    for k in k_vals:
        if k <= is_gt_tensor.shape[1]:
            row[f"R@{k}"] = float(is_gt_tensor[:, :k].any(dim=1).float().mean()) * 100
    return row


fig, axes = plt.subplots(1, 2, figsize=(13, 4))
k_vals = (1, 5, 10)

for ax, direction, title in [
    (axes[0], "i2t", "I2T  (image → text)"),
    (axes[1], "t2i", "T2I  (text → image)"),
]:
    clip_rec = compute_recall(clip_base[direction]["is_gt"], k_vals)
    for k in k_vals:
        key = f"R@{k}"
        if key in clip_rec:
            ax.axhline(
                clip_rec[key], linestyle="--", alpha=0.6,
                label=f"CLIP {key}={clip_rec[key]:.1f}%",
            )

    epochs = [s["epoch"] for s in snaps]
    for k in k_vals:
        key = f"R@{k}"
        vals = [compute_recall(s[direction]["is_gt"], k_vals).get(key) for s in snaps]
        if any(v is not None for v in vals):
            ax.plot(epochs, vals, marker="o", label=f"CoSiR {key}")

    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Recall (%)")
    ax.legend(fontsize=7, ncol=2)
    ax.grid(True, alpha=0.3)

fig.suptitle(
    f"Oracle R@K vs CLIP baseline  (fixed first-50 queries)  [{combine_side}-side]",
    fontsize=11,
)
plt.tight_layout()
plt.show()

---

## Top-N most improved I2T queries (CLIP → last epoch)

**Improvement score** = position of first GT hit in CLIP top-K **minus** position
in last-epoch top-K (both 0-indexed; `K` if GT not found in top-K).
A higher score means CoSiR moved the GT item much closer to rank 1 relative to CLIP.


In [ ]:
def gt_score(is_gt_list, K):
    """
    Scalar position score for one query's retrieved list.

    IMPROVEMENT_METRIC == "top1":
        0-indexed position of the first GT hit, or K if none found.
    IMPROVEMENT_METRIC == "mean":
        mean 0-indexed position of ALL GT hits, or K if none found.
    Both: lower is better, so improvement = clip_score - cosir_score.
    """
    positions = [i for i, v in enumerate(is_gt_list) if v]
    if not positions:
        return float(K)
    if IMPROVEMENT_METRIC == "mean":
        return float(np.mean(positions))
    else:  # "top1"
        return float(positions[0])


def top_improved_queries(direction: str, n: int = TOP_N_QUERIES):
    """
    Return list of (query_idx, clip_score, cosir_score, improvement_score)
    sorted by improvement descending, for the last available epoch snapshot.
    """
    last_snap = snaps[-1]
    K = clip_base[direction]["is_gt"].shape[1]
    n_queries = clip_base[direction]["is_gt"].shape[0]

    records = []
    for q in range(n_queries):
        clip_s = gt_score(clip_base[direction]["is_gt"][q].tolist(), K)
        cosir_s = gt_score(last_snap[direction]["is_gt"][q].tolist(), K)
        score = clip_s - cosir_s  # positive = CoSiR improved rank
        records.append((q, clip_s, cosir_s, score))

    records.sort(key=lambda x: -x[3])
    return records[:n]


last_snap = snaps[-1]
top_i2t = top_improved_queries("i2t", TOP_N_QUERIES)

print(
    f"Top-{TOP_N_QUERIES} most improved I2T queries  "
    f"(CLIP vs Epoch {last_snap['epoch']})  [metric: {IMPROVEMENT_METRIC}]"
)
metric_label = "rank" if IMPROVEMENT_METRIC == "top1" else "mean-rank"
print(
    f"{'Query':>6}  {f'CLIP {metric_label}':>14}  {f'CoSiR {metric_label}':>15}  {'Improvement':>12}"
)
print("-" * 54)
for q, cp, sp, score in top_i2t:
    K = clip_base["i2t"]["is_gt"].shape[1]
    cp_str = f"{cp+1:.1f}" if cp < K else f">{K}"
    sp_str = f"{sp+1:.1f}" if sp < K else f">{K}"
    print(f"{q:>6}  {cp_str:>14}  {sp_str:>15}  {score:>+12.2f}")

print()
for q, cp, sp, score in top_i2t:
    show_i2t(query_image_idx=q, snaps_override=[last_snap])

---

## Top-N most improved T2I queries (CLIP → last epoch)


In [ ]:
top_t2i = top_improved_queries("t2i", TOP_N_QUERIES)

print(
    f"Top-{TOP_N_QUERIES} most improved T2I queries  "
    f"(CLIP vs Epoch {last_snap['epoch']})  [metric: {IMPROVEMENT_METRIC}]"
)
metric_label = "rank" if IMPROVEMENT_METRIC == "top1" else "mean-rank"
print(
    f"{'Query':>6}  {f'CLIP {metric_label}':>14}  {f'CoSiR {metric_label}':>15}  {'Improvement':>12}"
)
print("-" * 54)
for q, cp, sp, score in top_t2i:
    K = clip_base["t2i"]["is_gt"].shape[1]
    cp_str = f"{cp+1:.1f}" if cp < K else f">{K}"
    sp_str = f"{sp+1:.1f}" if sp < K else f">{K}"
    print(f"{q:>6}  {cp_str:>14}  {sp_str:>15}  {score:>+12.2f}")

print()
for q, cp, sp, score in top_t2i:
    show_t2i(query_text_idx=q, snaps_override=[last_snap])